# Maximise LLM prompt-cache hits

Same cells as the [llm-sql docs page](./). Run top to bottom in a notebook with Python 3.12+.

You supply a seed, an evaluator, and a proposer. `meta.improve()` scores the GRG seed, asks a model to rewrite it, and keeps the best measured program.

Scoring the seed needs no API key and should print about `0.6961`. The improve loop needs `OPENROUTER_API_KEY`. One recorded run: `0.6954 → 0.7250 → evaluation_failed → 0.7268`.


## 1. Install


In [ ]:
%pip install "pandas==2.2.3" "numpy<2.1" networkx "meta-evolve @ git+https://github.com/sentient-xyz/meta-evolve.git"


## 2. Fetch the seed


In [ ]:
import os, subprocess
from pathlib import Path

import pandas as pd
import meta_evolve as meta

assert pd.__version__.startswith("2.2"), pd.__version__

WORK = Path("adrs_llm_sql_run").resolve()
TASK = WORK / "adrs-upstream" / "openevolve" / "examples" / "ADRS" / "llm_sql"
WORK.mkdir(parents=True, exist_ok=True)

if not TASK.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout",
         "https://github.com/UCB-ADRS/ADRS.git", str(WORK / "adrs-upstream")],
        check=True,
    )
    subprocess.run(
        ["git", "sparse-checkout", "set", "openevolve/examples/ADRS/llm_sql"],
        cwd=WORK / "adrs-upstream",
        check=True,
    )
    subprocess.run(["git", "checkout"], cwd=WORK / "adrs-upstream", check=True)

SEED = (TASK / "initial_program.py").read_text()
print(f"seed: {TASK / 'initial_program.py'} ({len(SEED)} chars)")


## 3. Measure it

This should print a `combined_score` near `0.6961` for GRG with no API key. It downloads `score_one.py` if that file is not next to the notebook.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from urllib.request import urlretrieve

GRADER = Path("score_one.py")
if not GRADER.exists():
    urlretrieve(
        "https://sentient-xyz.github.io/meta-evolve-docs/applications/llm-sql/score_one.py",
        GRADER,
    )

def evaluate(source):
    """Score one Evolved module. Returns combined_score, or raises on failure."""
    work = WORK / "candidates"
    work.mkdir(parents=True, exist_ok=True)
    program = work / "candidate.py"
    program.write_text(str(source))
    done = subprocess.run(
        [sys.executable, str(GRADER.resolve()), str(program)],
        cwd=str(WORK),
        env={**os.environ, "ADRS_TASK_DIR": str(TASK)},
        capture_output=True,
        text=True,
        timeout=900,
    )
    report = None
    for line in reversed(done.stdout.splitlines()):
        line = line.strip()
        if line.startswith("{"):
            report = json.loads(line)
            break
    if not report or not report.get("ok"):
        error = (report or {}).get("error") or done.stderr[-500:] or "score failed"
        raise RuntimeError(error)
    return report["combined_score"]

score = evaluate(SEED)
print(f"seed combined_score: {score:.4f}")
# Output:
# seed combined_score: 0.6961


## 4. Propose a rewrite


In [ ]:
import json, os, re, urllib.request

API_KEY = os.environ["OPENROUTER_API_KEY"]
MODEL = "anthropic/claude-sonnet-5"
SYSTEM = """You evolve a Python algorithm for the ADRS llm_sql task.
Reorder columns and rows only. Never change a cell.
Emit one module that defines class Evolved(Algorithm) with:
    def reorder(self, df, early_stop=0, row_stop=None, col_stop=None,
                col_merge=[], one_way_dep=[], distinct_value_threshold=0.8,
                parallel=True)
Import the base class with `from solver import Algorithm`.
Return only a ```python block."""

def propose(parent):
    payload = json.dumps({
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": "Improve this Evolved algorithm.\n\n```python\n" + str(parent) + "\n```"},
        ],
    }).encode()
    request = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=payload,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=900) as response:
        body = json.load(response)
    text = body["choices"][0]["message"]["content"]
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    source = max(blocks, key=len).strip() if blocks else text
    if "class Evolved" not in source:
        raise RuntimeError("model did not return class Evolved")
    return source


## 5. Run the loop


In [ ]:
result = meta.improve(
    seed=SEED,
    proposer=propose,
    evaluator=evaluate,
    trials=3,
)


## 6. See what changed


In [ ]:
for number, attempt in enumerate(result.trials()):
    score = attempt.metrics.get("score")
    print(f"trial {number}: {score:.4f}" if isinstance(score, (int, float)) else f"trial {number}: {attempt.state}")

print("selected source:")
print(result.best().value)
# Output (one recorded Claude Sonnet 5 / OpenRouter run; later trials vary):
# trial 0: 0.6954
# trial 1: 0.7250
# trial 2: evaluation_failed
# trial 3: 0.7268
